# 01. Target Lightcurve Analysis

Load and inspect a real MuSCAT differential lightcurve produced by the
`prose2` photometry pipeline. You will look up a target's observing
history in `muscat.db`, load its per-band lightcurve CSVs, and plot the
flux alongside the observing conditions (airmass, seeing) that drive
systematic trends.

**Environment:** run with the `prose` conda kernel
(`conda activate prose`), which provides numpy/pandas/matplotlib.
Lightcurve files are read from `$MUSCAT_PROSE_DIR` (default `~/ql/prose`).

In [ ]:
import os
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_db():
    """Return a read-only path to a populated muscat.db, or None.

    Checks $MUSCAT_DB_PATH first, then the repo-root copy one level up,
    then the working directory. A candidate only counts if its `targets`
    table actually has rows (skips empty schema stubs).
    """
    candidates = []
    if os.environ.get("MUSCAT_DB_PATH"):
        candidates.append(Path(os.environ["MUSCAT_DB_PATH"]))
    candidates += [Path("../muscat.db"), Path("muscat.db")]
    for p in candidates:
        if not p.exists():
            continue
        try:
            con = sqlite3.connect(f"file:{p}?mode=ro", uri=True)
            n = con.execute("SELECT count(*) FROM targets").fetchone()[0]
            con.close()
        except sqlite3.Error:
            continue
        if n > 0:
            return p.resolve()
    return None


def prose_dir():
    """Root of prose2 lightcurve output: $MUSCAT_PROSE_DIR or ~/ql/prose."""
    return Path(os.environ.get("MUSCAT_PROSE_DIR", Path.home() / "ql" / "prose"))


DB = find_db()
print("muscat.db:", DB if DB else "not found (metadata cells will be skipped)")
print("prose dir:", prose_dir())

## Choose an observation

The default is a four-band MuSCAT3 observation of TOI-5191. Edit these to
any target/instrument/date present in your prose output directory.

In [ ]:
TARGET = "TOI5191"
INSTRUMENT = "muscat3"
DATE = "230521"
BIN_MINUTES = 5

## 1. Look up the target in the observing log

The `targets` table is a per-object rollup: which instruments and filters
observed it, on which dates, and how much total exposure time. This is the
fastest way to see what data exist before loading anything.

In [ ]:
if DB is not None:
    con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
    con.row_factory = sqlite3.Row
    row = con.execute(
        "SELECT object, instruments, filters, dates, n_dates, n_frames, "
        "total_exptime, ra, declination FROM targets WHERE object = ?",
        (TARGET,),
    ).fetchone()
    con.close()
    if row:
        for k in row.keys():
            print(f"{k:14s}: {row[k]}")
    else:
        print(f"{TARGET} is not in the targets table; check the spelling.")
else:
    print("muscat.db not found; skipping the metadata lookup.")

## 2. Load the per-band lightcurves

Each `prose2` run writes one CSV per filter. The `Flux` column is the
differential (target / comparison) flux; `Err` is its formal uncertainty.
The remaining columns (`Airmass`, `FWHM(pix)`, `Bkg(ADU)`, `Dx/Dy`) are
per-frame covariates used later for detrending.

In [ ]:
def load_band_csvs(target, instrument, date):
    """Load every per-band prose2 lightcurve CSV for one observation.

    Files are named <target>_<instrument>_<band>_<date>.csv and live
    under <prose_dir>/<instrument>/<date>/. Returns {band: DataFrame}.
    """
    night = prose_dir() / instrument / date
    out = {}
    if not night.is_dir():
        return out
    pat = re.compile(rf"^{re.escape(target)}_{re.escape(instrument)}_(.+)_{re.escape(date)}$")
    for csv in sorted(night.glob(f"{target}_{instrument}_*_{date}.csv")):
        m = pat.match(csv.stem)
        if m:
            out[m.group(1)] = pd.read_csv(csv)
    return out


# Preferred display order for the standard g, r, i, z filter set.
_BAND_ORDER = ["gp", "g", "rp", "r", "ip", "i", "zs", "z_s", "zp", "z"]


def order_bands(bands):
    known = [b for b in _BAND_ORDER if b in bands]
    rest = [b for b in bands if b not in known]
    return known + sorted(rest)

In [ ]:
curves = load_band_csvs(TARGET, INSTRUMENT, DATE)
if not curves:
    print(f"No lightcurve CSVs found for {TARGET} ({INSTRUMENT} {DATE}) under {prose_dir()}.")
    print("Set MUSCAT_PROSE_DIR to your prose output root, or edit TARGET/INSTRUMENT/DATE above.")
else:
    bands = order_bands(list(curves))
    print(f"Loaded {len(bands)} band(s): {bands}")
    for b in bands:
        df = curves[b]
        rms_ppt = np.std(df["Flux"]) * 1e3
        span_h = (df["BJD_TDB"].iloc[-1] - df["BJD_TDB"].iloc[0]) * 24
        print(f"  {b:4s}  {len(df):4d} pts  {span_h:.2f} h  rms = {rms_ppt:.2f} ppt")

## 3. Plot the multi-band lightcurves

Bands are offset vertically so all filters are visible at once. Observing
in several colors at once is the point of MuSCAT: a real astrophysical
transit is achromatic (same depth in every band), while stellar activity
or blends are usually color-dependent.

In [ ]:
if curves:
    bands = order_bands(list(curves))
    t0 = min(df["BJD_TDB"].iloc[0] for df in curves.values())
    step = 0.04  # vertical offset between bands, in relative flux

    fig, ax = plt.subplots(figsize=(9, 6))
    for i, b in enumerate(bands):
        df = curves[b]
        t = (df["BJD_TDB"] - t0) * 24
        offset = -i * step
        ax.scatter(t, df["Flux"] + offset, s=6, alpha=0.6, label=b)
        ax.axhline(1.0 + offset, color="0.6", lw=0.7, ls="--", zorder=0)
        ax.text(t.max() + 0.03, 1.0 + offset, b, va="center", fontsize=9)

    ax.set_xlabel("Time from first frame (hours)")
    ax.set_ylabel("Relative flux (bands offset for clarity)")
    ax.set_title(f"{TARGET}  {INSTRUMENT}  {DATE}")
    fig.tight_layout()
    plt.show()

## 4. Bin one band

Binning in time averages down white noise: the scatter of the binned
points falls roughly as 1/sqrt(N) per bin. This is how a shallow transit
becomes visible above the per-frame scatter.

In [ ]:
def bin_time(t, y, bin_minutes):
    """Bin (t in days, y) into fixed time bins; return centers, means, SEM."""
    width = bin_minutes / (24 * 60)
    edges = np.arange(t.min(), t.max() + width, width)
    idx = np.digitize(t, edges)
    centers, means, errs = [], [], []
    for k in range(1, len(edges)):
        m = idx == k
        if m.sum() == 0:
            continue
        centers.append(t[m].mean())
        means.append(y[m].mean())
        errs.append(y[m].std(ddof=1) / np.sqrt(m.sum()) if m.sum() > 1 else 0.0)
    return np.array(centers), np.array(means), np.array(errs)


if curves:
    band = order_bands(list(curves))[0]
    df = curves[band]
    t = ((df["BJD_TDB"] - df["BJD_TDB"].iloc[0]) * 24).to_numpy()
    y = df["Flux"].to_numpy()
    tb, yb, eb = bin_time(t, y, BIN_MINUTES)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(t, y, s=6, color="0.7", label="unbinned")
    ax.errorbar(tb, yb, yerr=eb, fmt="o", ms=5, color="C3", capsize=2,
                label=f"{BIN_MINUTES:.0f}-min bins")
    ax.set_xlabel("Time from first frame (hours)")
    ax.set_ylabel("Relative flux")
    ax.set_title(f"{TARGET} {band}: binning improves per-point precision")
    ax.legend()
    fig.tight_layout()
    plt.show()

## 5. Flux vs. observing conditions

Before trusting any dip in the flux, check it against the covariates. A
trend that tracks airmass or a seeing (FWHM) spike is instrumental, not
astrophysical. The transit-fitting stage (notebook 03 and `transit_fit.py`)
models these covariates explicitly.

In [ ]:
if curves:
    band = order_bands(list(curves))[0]
    df = curves[band]
    t = (df["BJD_TDB"] - df["BJD_TDB"].iloc[0]) * 24

    fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
    axes[0].scatter(t, df["Flux"], s=6, color="C0")
    axes[0].set_ylabel("Relative flux")
    axes[1].plot(t, df["Airmass"], color="C1")
    axes[1].set_ylabel("Airmass")
    fwhm_col = "FWHM(pix)" if "FWHM(pix)" in df.columns else None
    if fwhm_col:
        axes[2].plot(t, df[fwhm_col], color="C2")
        axes[2].set_ylabel("FWHM (pix)")
    axes[2].set_xlabel("Time from first frame (hours)")
    axes[0].set_title(f"{TARGET} {band}: flux vs. observing conditions")
    fig.tight_layout()
    plt.show()

## Summary

- `muscat.db` answers *what was observed*; the prose CSVs hold the actual
  photometry.
- The `Flux` column is a differential lightcurve; `Err` is its uncertainty.
- Multi-band data separate real (achromatic) transits from color-dependent
  systematics.
- Always inspect flux against airmass and seeing before interpreting it.

Next: **02** shows how that `Flux` column is measured from raw images.